#### 1. Setting up the environment 

Where we create .env file with all the environment variables that are required for this to run

In [4]:
from langsmith import utils

# check and print whether LangSmith tracing is currently enabled
print(f"LangSmith tracing is enabled: {utils.tracing_is_enabled()}")

LangSmith tracing is enabled: False


#### 2. Choosing our Dataset: 

- Here we are choosing Chinook Dataset, which is popular sample dataset used for learning and testing SQL
- It contains a digital music store's data and operations, such as customer info, purchase history & music catalog
- It comes in multiple formats like MySQL, PostgreSQL etc.. But here I'm using SQLite version of the data

Let's define a function that will set up the SQLite database for us

In [7]:
import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

def get_engine_for_chinook_db():
    """
    Pull SQL file, populate in-memory database, and create engine,

    Downloads the Chinook dataset SQL script from GitHub and creates an in-memory
    SQLite database populated with sample data.

    Returns:
        sqlalchemy.engine.Engine: SQLAlchemy engine connected to the in-memory database
    """

    # Download the Chinook Dataset SQL script from the official repo
    url = 'https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql'
    response = requests.get(url)
    sql_script = response.text

    # Create an in-memory SQLite database connection
    # check_same_thread=False allows the connection to be used across threads
    connection = sqlite3.connect(":memory:", check_same_thread=False)

    # Execute the SQL script to populate the database with sample data
    connection.executescript(sql_script)

    # create and return a SQLAlchemy engine that uses the populated connection
    return create_engine(
        "sqlite://",    # SQLite URL scheme
        creator=lambda: connection,  # Function that returns the database connection
        poolclass=StaticPool,   # Use StaticPool to maintain single connection
        connect_args={'check_same_thread': False},  # Allow cross-thread usage  
        )


- So we just defined our 1st function, get_engine_for_chinook_db(), which sets up a temporary in-memory SQLite database using the Chinook sample dataset
- It downloads the SQL script from GitHub, creates the database in memory, runs the script to populate it with tables and data, and then returns a SQLAlchemy engine connected to this database
- Now, we need to intialize this func so that SQLite database gets created

In [8]:
# intialize the database engine with the Chinook sample data
engine = get_engine_for_chinook_db()

# create a LangChain SQLDatabase wrapper around the engine
# This provides convenient method for database operations and query execution
db = SQLDatabase(engine)

#### 3. Short-Term and Long-Term Memory

In LangGraph, we differentiate b/w short-term memory and long-term memory. Here is a quick diff:

- Short-term memory helps an agent keep track of the current conversation. In LangGraph, this is handled by a **MemorySaver**, which saves and resumes the state of the conversation.
- While long-tern memory lets the agent remeber info across different conversations, like user preferences. For ex: we can use an **InMemoryStore** for quick storage, but in real apps, you'd use a more permanent database


In [9]:
from langgraph.checkpoint.memory import MemorySaver     # for short-term
from langgraph.store.memory import InMemoryStore        # for long-term

# Initialize long-term memory store for persistent data b/w conversations
in_memory_store = InMemoryStore()

# Initialize checkpointer for short-term memory within a single thread/conversation
checkpointer = MemorySaver()

#### 4. Our Multi-Agent Architecture

- We will start with ReAct agent and additional steps into the workflow, simulating a realistic customer support ex, showcasing human-in-the-loop, long term memory, and LangGraph pre-built library

![Multi-agent architecture](images/multi-agent-architecture.png)


Our workflow starts with:
- 1. **human_input**: where the user provides account info
- 2. Then, in **verify_info**, the system checks the account and clarifies the user's intent if needed.
- 3. Next, **load_memory** retrieve's the user's music preferences
- 4. The **supervisor** coordinates 2 sub-agents: **music_catalog** (for music data) and **invoice_info** (for billing)
- 5. Finally, **create_memory** updates the user's memory with new info from the interaction

In [14]:
print(db.get_table_info())
print(db.get_usable_table_names())


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated, List
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.managed.is_last_step import RemainingSteps


class State(TypedDict):
    """
    State schema for the multi-agent customer support workflow

    This defines the shared data structure that flows between nodes in the graph, 
    representing the current snapshot of the conversation and agent state.
    """

    # Customer identifier retrived from the account verification
    customer_id: str

    # Conv history with automatic message aggregation
    messages: Annotated[List[AnyMessage], add_messages]

    # User preferences & context loaded from long-term memory store
    loaded_memory: str

    # Counter to prevent infinte recursion in agent workflow
    remaining_steps: RemainingSteps

- This State class serve as the blueprint for how info is managed and passed b/w different parts of our mult-agent system
- Next, we will extend the agent's abilities using **Tools**
- For our agent, tools will connect to the **Chinook dataset** to fetch music-realted info

In [ ]:
from langchain_core.tools import tool
import ast

@tool
def get_albums_by_artist(artist: str):
    """
    Get albums by an artist from the music database.

    Args:
        artist (str): The name of the artist to search for albums

    Returns:
        str: Database query results containing album titles and artist names
    """

    return db.run(
        f""" 
        SELECT Album.Title, Artist.Name
        FROM Album
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE '%{artist}%';
        """
    )

@tool
def get_tracks_by_artist(artist: str):
    """ 
    Get songs/tracks by an artist (or similar artists) from the music database

    Args:
        artist (str): The name of the artist to search for tracks
    
    Returns:
        str: Database query results containing song names and artist names
    """

    return db.run(
        f""" 
        SELECT Track.Name as SongName, Artist.Name as ArtistName
        FROM Album
        LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId
        LEFT JOIN Track ON Track.AlbumId = Album.AlbumId
        WHERE Artist.Name LIKE '%{artist}%';
        """,
        include_columns=True
    )

@tool
def get_songs_by_genre(genre: str):
    """
    Fetch songs from the database that match a specific genre.

    This function first looks up the genre ID(s) for the given genre name, 
    then retrives songs that belong to those genre(s), limiting results 
    to 8 songs grouped by artist.

    Args:
        genre (str): The genre of the songs to fetch

    Returns:
        list[dict] or str: A list of songs with artist information that match
                        the specified genre, or an error message if no songs found.
    """

    # First, get the genre ID(s) for the specified genre
    genre_id_query = f"SELECT GenreId FROM Genre WHERE Name LIKE '%{genre}%"
    genre_ids = db.run(genre_id_query)

    # Check if any genres were found
    if not genre_ids:
        return f"No songs found for the genre: {genre}"

    # Parse the genre IDs and format them for the SQL query
    genre_ids = ast.literal_eval(genre_ids)
    genre_id_list = ", ".join(str(gid[0]) for gid in genre_ids)

    # Query for songs in the specified genre(s)
    songs_query = f""" 
    SELECT Track.Name as SongName, Artist.Name as ArtistName
    FROM Track
    LEFT JOIN Album ON Track.AlbumId = Album.AlbumId
    LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId
    WHERE Track.GenreId IN ({genre_id_list})
    GROUP BY Artist.Name
    LIMIT 8
    """

    songs = db.run(songs_query, include_columns=True)

    # Check if any songs were found
    if not songs:
        return f"No songs found for the genre: {genre}"

    # Format the results into a structured list of dictionaries
    formatted_songs = ast.literal_eval(songs)
    return [
        {"Song": song['SongName'], 'Artist': song['ArtistName']}
        for song in formatted_songs
    ]

@tool
def check_for_songs(song_title):
    """ 
    Check if a song exists in the database by its name

    Args: 
        song_title (str): The title of the song to search for.

    Returns:
        str: Database query results containing all track information 
                for songs matching the given title
    """

    return db.run(
        f""" 
        SELECT * from Track WHERE Name LIKE '%{song_title}%';
        """,
        include_columns=True
    )